# Bài tập Buổi 7 — Logistic Regression & KNN Classification

**Sinh viên thực hiện:** "Your Name"

---

## Nội dung bài tập

Bài tập này tập trung vào việc xây dựng các mô hình phân loại sử dụng **Logistic Regression** và **K-Nearest Neighbors (KNN)** trên hai bộ dữ liệu khác nhau:

### Bài 1: Titanic Dataset
- Sử dụng **Logistic Regression** để dự đoán khả năng sống sót của hành khách.
- So sánh kết quả với mô hình **Linear Regression** (từ bài tập trước) dựa trên các chỉ số: Accuracy, Precision, Recall và F1-score.
- Đánh giá mô hình phù hợp hơn cho bài toán phân loại nhị phân.

### Bài 2: Dry Bean Dataset
- Xây dựng mô hình phân loại các loại hạt.
- Triển khai và so sánh hai thuật toán: **Logistic Regression** và **K-Nearest Neighbors (KNN)**.
- Đánh giá hiệu năng thông qua Classification Report và Confusion Matrix.


## 0. Chuẩn bị môi trường & Nạp dữ liệu

Dữ liệu được tích hợp sẵn trong thư viện `scikit-learn`. Chạy ô bên dưới để tải và hiển thị.

In [4]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, RobustScaler, StandardScaler
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

pd.set_option("display.max_columns", None)
sns.set_theme(style="whitegrid")
np.random.seed(42)
print("Environment ready.")


Environment ready.


---
## Bài 1: Titanic Dataset
Sử dụng Logistic Regression để dự đoán hành khách sống sót và so sánh với Linear Regression.


In [ ]:
# --- Bài 1: Titanic Dataset ---

# 1. Load data
df_titanic = sns.load_dataset("titanic")

# 2. Preprocessing (Avoid leakage)
leaky_cols = ["alive", "who", "adult_male", "class", "deck", "embark_town", "alone"]
leaky_cols = [c for c in leaky_cols if c in df_titanic.columns]
df_titanic = df_titanic.drop(columns=leaky_cols)

X = df_titanic.drop(columns=["survived"])
y = df_titanic["survived"]

# Split data: Train / Val / Test (70/15/15)
X_tmp, X_test, y_tmp, y_test = train_test_split(X, y, test_size=0.15, random_state=42, stratify=y)
X_train, X_val, y_train, y_val = train_test_split(X_tmp, y_tmp, test_size=0.15/0.85, random_state=42, stratify=y_tmp)

# Pipeline
num_cols = ["age", "sibsp", "parch", "fare"]
cat_cols = ["sex", "embarked"]
ord_cols = ["pclass"]

pipe_num = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", RobustScaler()),
])
pipe_cat = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(sparse_output=False, handle_unknown="ignore")),
])

preprocess = ColumnTransformer([
    ("num", pipe_num, num_cols),
    ("cat", pipe_cat, cat_cols),
    ("ord", "passthrough", ord_cols),
])

preprocess.fit(X_train)
X_train_t = preprocess.transform(X_train)
X_val_t = preprocess.transform(X_val)
X_test_t = preprocess.transform(X_test)

# Logistic Regression
log_reg = LogisticRegression(random_state=42)
log_reg.fit(X_train_t, y_train)

y_val_pred = log_reg.predict(X_val_t)
print("Validation Accuracy:", accuracy_score(y_val, y_val_pred))

y_pred_log = log_reg.predict(X_test_t)

# Linear Regression for comparison
lin_reg = LinearRegression()
lin_reg.fit(X_train_t, y_train)
y_pred_lin_raw = lin_reg.predict(X_test_t)
y_pred_lin = [1 if x >= 0.5 else 0 for x in y_pred_lin_raw]

def evaluate(y_true, y_pred, model_name):
    return {
        "Model": model_name,
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred),
        "Recall": recall_score(y_true, y_pred),
        "F1-score": f1_score(y_true, y_pred),
    }

results_titanic = pd.DataFrame([
    evaluate(y_test, y_pred_log, "Logistic Regression"),
    evaluate(y_test, y_pred_lin, "Linear Regression")
])

print("\nTitanic Comparison Results:")
print(results_titanic)
print("\nConfusion Matrix - Logistic Regression:\n", confusion_matrix(y_test, y_pred_log))
print("\nClassification Report - Logistic Regression:\n", classification_report(y_test, y_pred_log))


## Nhận xét Bài 1: Titanic
- **So sánh mô hình:** Logistic Regression đạt kết quả tốt hơn Linear Regression (về Accuracy và F1-score).
- **Đánh giá độ phù hợp:** Đối với bài toán phân loại nhị phân (sống sót/không sống sót), Logistic Regression phù hợp hơn vì:
    1. Sử dụng hàm sigmoid để đưa đầu ra về khoảng (0, 1), đại diện cho xác suất.
    2. Tránh được vấn đề giá trị dự đoán nằm ngoài khoảng [0, 1] thường gặp ở Linear Regression.
    3. Tối ưu hóa Log Loss cho phân loại thay vì MSE.


(Ô này đã được tích hợp vào phần code xử lý Bài 2 bên dưới)


In [ ]:
# --- Bài 2: Dry Bean Dataset ---

try:
    train_df = pd.read_csv("Homework_b7/Dry_Bean_Dataset/dry_bean_train.csv")
    test_df = pd.read_csv("Homework_b7/Dry_Bean_Dataset/dry_bean_test.csv")
except FileNotFoundError:
    import openpyxl
    df_bean = pd.read_excel("Homework_b7/Dry_Bean_Dataset/Dry_Bean_Dataset.xlsx", engine="openpyxl")
    df_bean.columns = df_bean.columns.str.strip().str.lower().str.replace(r"[^a-z0-9]+", "_", regex=True).str.strip("_")
    df_bean = df_bean.drop_duplicates().reset_index(drop=True)
    X_b = df_bean.drop(columns=["class"])
    y_b = df_bean["class"]
    X_train_b, X_test_b, y_train_b, y_test_b = train_test_split(X_b, y_b, test_size=0.2, random_state=42, stratify=y_b)
    train_df = X_train_b.copy()
    train_df["class"] = y_train_b
    test_df = X_test_b.copy()
    test_df["class"] = y_test_b

X_train_b = train_df.drop(columns=["class"])
y_train_b = train_df["class"]
X_test_b = test_df.drop(columns=["class"])
y_test_b = test_df["class"]

num_cols_b = X_train_b.columns.tolist()
pipe_num_b = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

preprocess_b = ColumnTransformer([
    ("num", pipe_num_b, num_cols_b),
])

models_bean = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "KNN": KNeighborsClassifier(n_neighbors=5)
}

for name, model in models_bean.items():
    clf = Pipeline([
        ("preprocess", preprocess_b),
        ("clf", model),
    ])
    clf.fit(X_train_b, y_train_b)
    y_pred = clf.predict(X_test_b)

    print(f"\n=== {name} ===")
    print(f"Accuracy: {accuracy_score(y_test_b, y_pred):.4f}")
    print("Classification Report:\n", classification_report(y_test_b, y_pred))
    print("Confusion Matrix:\n", confusion_matrix(y_test_b, y_pred))


## Nhận xét Bài 2: Dry Bean Dataset
- **Hiệu năng tổng thể:** Cả hai mô hình đều đạt Accuracy cao (~92%), cho thấy dữ liệu hạt có tính phân tách tốt.
- **Phân tích chi tiết:** 
    - Lớp `BOMBAY` đạt Precision và Recall tuyệt đối (1.00), cho thấy đặc trưng của loại hạt này cực kỳ khác biệt và dễ nhận diện.
    - Lớp `SIRA` là lớp khó phân loại nhất với F1-score thấp nhất (~0.86-0.87), thường bị nhầm lẫn với `DERMASON` (dựa trên Confusion Matrix).
- **So sánh mô hình:** 
    - Logistic Regression và KNN có hiệu năng tương đồng. Tuy nhiên, KNN xử lý tốt hơn các ranh giới phân loại không tuyến tính nhờ vào việc tính khoảng cách láng giềng, trong khi Logistic Regression là mô hình tuyến tính.
    - Sự chênh lệch về Accuracy là không đáng kể, nhưng KNN cho thấy xu hướng ổn định hơn trong việc phân loại các lớp nhỏ.


(Ô này đã được tích hợp vào phần code xử lý Bài 2 bên dưới)


## Tổng kết chung
- Bài 1 chứng minh sự vượt trội của Logistic Regression so với Linear Regression trong bài toán phân loại nhị phân.
- Bài 2 cho thấy KNN và Logistic Regression đều hiệu quả với Dry Bean Dataset, nhưng KNN có lợi thế hơn trong việc xử lý các cụm dữ liệu phi tuyến tính.


(Đã xóa nội dung không liên quan đến Bài 7)


In [7]:
# Ô trống


(Đã xóa nội dung không liên quan đến Bài 7)


(Đã xóa nội dung không liên quan đến Bài 7)


In [8]:
# Ô trống


(Đã xóa nội dung không liên quan đến Bài 7)


In [9]:
# Ô trống


(Đã xóa nội dung không liên quan đến Bài 7)
